## Initial Design

main components for course recommendation system

1. Eligibility_pipeline - Based on the student academic records such as school, competetive exams, UG / PG records etc.. data will be used to check eligibility from the courses that university provides.

2. Similarity_search_model - cosine similarity engine to search for relevant courses among the eligible courses. Then rank the Top-K courses as the output


## Current action plan

First the course data will be required in the structured format to store it as vector embedding, which will be used for similarity search.

Course data will be structured as follows:
```python
1. course_id
2. course_name
3. degree_level        # UG / PG / PhD
4. domain              # Engineering / Management /5. Science / Humanities

eligible_degrees    # list-like string
required_subjects   # list-like string

required_exam       # JEE / CAT / GATE / NET / NONE
min_exam_score
min_gpa

description         # for TF-IDF
keywords            # comma separated
career_outcomes     # optional but useful
```


## First Implementation

The first code implemetation originated from the need of a clean dataset for the eligibility pipeline. 

The current dataset had the following fields.
1. Program Name
2. Program level
3. Name of Faculty
4. Eligibility criteria

There were more fields in the datset but for the project's scope these are relevant.

The restructuring pipeline [restructure.ipynb](../../restructure_pipeline/restructure.ipynb) has the complete implementation.
The pipeline is a small Agentic workflow made with langchain-ollama library uses a locally running model `llama3.2` to generate a structure output which then will be handled by the `pandas` library to generate the desired dataset.

The ouput dataset will have the following fields
1. program_name: str 
2. program_level: str 
3. domain: str 
4. eligibility: str 
5. description: str 
6. skills_learned: List[str]
7. career_outcomes: List[str] 

The structure output is genrate by the model using pydantic model to ensure type safety. Dateset with these dimensions will be used for eligibility criteria and ML similarity to generate recommendations.

## Eligibility Pipeline

The dataset has been finalised and is ready for eligibility pipeline. This pipeline will serve the purpose of filteriing all the eligible course for a student profile. The filtered courses then will be matched against the student's profile in a cosine similarity engine.

The finalised dataset has the following schema:-
```python
class EligibilityStruct(BaseModel):
    min_degree_level: Optional[str] = Field(
        default=None,
        description="One of: PreUG, UG, PG, PhD"
    )
    min_marks_general: Optional[float] = Field(
        default=None,
        description="Minimum percentage for general category",
        ge=0.0,
        le=100.0
    )
    min_marks_reserved: Optional[float] = Field(
        default=None,
        description="Minimum percentage for reserved category (SC/ST/OBC/PwD)",
        ge=0.0,
        le=100.0
    )

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )
    eligibility_struct: EligibilityStruct = Field(description="Eligibiliy in the structured format")
```

The dataset schema is validates using pydantic models ensuring type-safety during runtime.